# Experiments notebook
everything that didnt classify in the main Final Catalog notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
from scipy import stats
import astropy

## Analyzing the Catalog Dataframe

In [ ]:
# ==============================================================================
# 1. Load the Master Catalog
# ==============================================================================
data_path = '../data/extra_catalogs/'
df_master = pd.read_csv(data_path + 'K2_Kepler_TESS_x_GALAH_LAMOST.csv')


#Primer objeto 
df_master.iloc[0] 
# this returns a pandas Series object, which is a one-dimensional array-like object 
# that can hold any data type. It has an index and a name, and it can be thought of as a single column 
# of a DataFrame. In this case, it returns the first row of the DataFrame df_master, with the index being
# the column names and the values being the corresponding values for that row.

print(f"Value at [0, 0]: {df_master.iloc[0, 0]}") # returns just the value 

print(f"Value at [0, 'K2_campaign']: {df_master.loc[0, 'K2_campaign']}")  # This will return the value in the 'K2_ID' column for the first row 

In [ ]:
df_master.iloc[[0],[0]] #this returns a DataFrame with a single value, rather than a Series. The double brackets indicate that we want to select a subset of the DataFrame, rather than a single row or column.


In [ ]:
df_master.loc[0:10, ['Dnu']]

In [ ]:
"""

# ==============================================================================
# Some trials to select fe to prepare Unified Metallicity (with no clean mask)
# ==============================================================================

# First some trials
df_master['feh'] = np.nan
df_master.loc[galah_mask, 'feh'] = pd.to_numeric(df_master.loc[galah_mask, 'fe_h_2'], errors='coerce')
df_master.loc[lamost_mask, 'feh'] = pd.to_numeric(df_master.loc[lamost_mask, '__Fe_H_'], errors='coerce')
df_master['feh'] = df_master['feh'].fillna(pd.to_numeric(df_master['Fe_H_1'], errors='coerce')) #If I want to use the Fe_H_1 (from APOGGE and GALAH) column as a backup for missing values, I can uncomment this line. It will fill in any NaN values in the 'feh' column with the corresponding values from the 'Fe_H_1' column, if they exist. 


df_master['feh2'] = np.nan
df_master.loc[galah_mask, 'feh2'] = pd.to_numeric(df_master.loc[galah_mask, 'fe_h_2'], errors='coerce')
df_master.loc[lamost_mask, 'feh2'] = pd.to_numeric(df_master.loc[lamost_mask, '__Fe_H_'], errors='coerce')

df_master['feh3'] = np.nan
df_master.loc[galah_mask, 'feh3'] = df_master.loc[galah_mask, 'fe_h_2']
df_master.loc[lamost_mask, 'feh3'] = df_master.loc[lamost_mask, '__Fe_H_']

# Check if feh and feh2 are identical (same values and same NaN/blank positions)

fe_p = 'feh'
fe_s = 'feh3'
same_feh = df_master[fe_p].equals(df_master[fe_s])
print(f"feh and feh2 are identical: {same_feh}")

if not same_feh:
    mask_diff = (
        (df_master[fe_p].isna() != df_master[fe_s].isna())
        | (
            df_master[fe_p].notna()
            & df_master[fe_s].notna()
            & (df_master[fe_p] != df_master[fe_s])
        )
    )
    print(df_master.loc[mask_diff, [fe_p, fe_s]].head())
    print(f"Number of rows with different values in 'feh' and 'feh2': {mask_diff.sum()} / {len(df_master)}")
"""


## Studying the population

##### Toomre diagram Cartesian coord

In [ ]:
"""
from astropy.coordinates import SkyCoord, Galactocentric
import astropy.units as u

# when cross matching sometimes I put in the same column two different things, maybe I will change this 
# in the future, so I only have to change the key and not everywhere in the code.

general_ra_key = 'ra' #Column originally from GALAH but also has LAMOST ra  
general_dec_key = 'dec' # same
general_telescope_ID = 'K2_ID'
general_2mass_ID_unformated = 'APOGEE_ID'
general_alpha_over_fe_or_metal = 'alpha_M'

cols_needed = [general_ra_key, general_dec_key, 'dist_Dnu', 'vR', 'vT', 'vZ']

# 1. Force everything in these columns to be numeric. Empty strings/text become NaN.
for col in cols_needed:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce')

# 2. Now the mask will successfully catch all the missing data
mask_kinematics = df_master[cols_needed].notna().all(axis=1)

mask_kinematics = mask_kinematics

# Pass the masked subset directly to your function/plot without saving a new variable
c = SkyCoord(
    ra=df_master.loc[mask_kinematics, general_ra_key].to_numpy() * u.deg, 
    dec=df_master.loc[mask_kinematics, general_dec_key].to_numpy() * u.deg, 
    distance=df_master.loc[mask_kinematics, 'dist_Dnu'].to_numpy() * u.kpc, 
    frame='icrs'
)


#We only have proper motion for GALAH, no LAMOST

gc = c.transform_to(Galactocentric())
phi = np.arctan2(gc.y.value, gc.x.value)


v_X = df_master.loc[mask_kinematics,'vR'].to_numpy()*np.cos(phi) - df_master.loc[mask_kinematics,'vT'].to_numpy()*np.sin(phi)
v_Y = df_master.loc[mask_kinematics,'vR'].to_numpy()*np.sin(phi) + df_master.loc[mask_kinematics,'vT'].to_numpy()*np.cos(phi)
v_Z = df_master.loc[mask_kinematics,'vZ'].to_numpy()

# Ploting

toomre_y = np.sqrt(v_X**2 + v_Z**2)
plt.ylim(0,300)
plt.xlim(-500,300)
plt.scatter(v_Y, toomre_y, s=2)
plt.xlabel(r'$v_Y$ [km/s]')
plt.ylabel(r'$\sqrt{v_X^2+v_Z^2}$ [km/s]')
plt.show()


df_sub= df_master.loc[mask_kinematics].copy()
df_sub['phi'] = phi  # from your code above

# Do you have a column distinguishing GALAH vs LAMOST rows? Use it here.
# If not, infer it from which ID columns are non-null, e.g.:
#is_galah = df_sub['APOGEE_ID'].notna()  # or whatever flags GALAH rows



plt.hist(df_sub.loc[galah_mask, 'ra'], bins=50, alpha=0.5, label='GALAH')
plt.hist(df_sub.loc[lamost_mask, 'ra'], bins=50, alpha=0.5, label='LAMOST')
plt.legend(); plt.xlabel('RA')
plt.show()

df_sub= df_master.loc[mask_kinematics].copy()
df_sub['phi'] = phi  # from your code above

# Do you have a column distinguishing GALAH vs LAMOST rows? Use it here.
# If not, infer it from which ID columns are non-null, e.g.:
#is_galah = df_sub['APOGEE_ID'].notna()  # or whatever flags GALAH rows



plt.hist(df_sub.loc[galah_mask, 'dec'], bins=50, alpha=0.5, label='GALAH')
plt.hist(df_sub.loc[lamost_mask, 'dec'], bins=50, alpha=0.5, label='LAMOST')
plt.legend(); plt.xlabel('DEC')
plt.show()
"""

In [ ]:
"""#Creating a filter for metallicity and age in order to clean the sample from Second Red clump stars

feh_lim = -0.5
age_lim = 2.5

age_conditions = [
    (mass > age_lim, f'Age > {age_lim} Gyr'),
    (mass <= age_lim, f'Age ≤ {age_lim} Gyr')
]
feh_conditions = [
    (feh > feh_lim, f'[Fe/H] > {feh_lim} dex'),
    (feh <= feh_lim, f'[Fe/H] ≤ {feh_lim} dex')
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)

for row, (age_condition, age_label) in enumerate(age_conditions):
    for col, (feh_condition, feh_label) in enumerate(feh_conditions):
        ax = axes[row, col]
        selection =  age_condition & feh_condition

        stars = df_master.loc[selection].dropna(
            subset=['Teff_kiel', 'logg_willet_l']
        )

        ax.scatter(
            stars['Teff_kiel'],
            stars['logg_willet_l'],
            s=12,
            alpha=0.4
        )
        ax.set_title(f'{age_label} and {feh_label}\nN={len(stars)}')
        ax.set_xlim(5400, 4100)
        ax.set_ylim(3.5, 1.0)
        ax.grid(True, alpha=0.3)

for ax in axes[1, :]:
    ax.set_xlabel(r'$T_{\mathrm{eff}}$ [K]')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\log\,g$ [dex]')

fig.suptitle('Kiel diagrams split by age and metallicity')
plt.tight_layout()
plt.show()"""

### Stellar parameters 

#### Analysis of [Fe/h]

In [ ]:
# ==============================================================================
# Using equal number of stars by bin
# ==============================================================================

n_bins = 6  # Change this to choose a different number of bins

feh_bin, bin_edges = pd.qcut(
    feh, q=n_bins, retbins=True, duplicates='drop'
)

counts = feh_bin.value_counts(sort=False)

print(f"Created {len(bin_edges) - 1} bins for {len(feh)} stars:")
for i, count in enumerate(counts):
    print(f"Bin {i + 1}: [{bin_edges[i]:.4f}, {bin_edges[i + 1]:.4f}] dex — {count} stars")

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
axes = axes.ravel()

for i, ax in enumerate(axes):
    lower, upper = bin_edges[i], bin_edges[i + 1]
    in_bin = feh.ge(lower) & (feh.le(upper) if i == len(axes) - 1 else feh.lt(upper))
    stars = df_master.loc[in_bin]
      
    ax.scatter(stars['Teff_kiel'], stars['logg_scaling'], s=12, alpha=0.5)
    ax.set_title(f'[{lower:.2f}, {upper:.2f}] dex — N={len(stars)}')
    ax.set_xlim(5400, 4100)
    ax.set_ylim(3.5, 1.0)
    ax.grid(True, alpha=0.3)

for ax in axes[3:]:
    ax.set_xlabel(r'$T_{\mathrm{eff}}$ [K]')
for ax in axes[::3]:
    ax.set_ylabel(r'$\log\,g$ [dex]')

fig.suptitle('Kiel diagrams by metallicity bin')
plt.tight_layout()
plt.show()


In [ ]:
# HR diagrams in the same metallicity bins
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
axes = axes.ravel()

for i, ax in enumerate(axes):
    lower, upper = bin_edges[i], bin_edges[i + 1]
    in_bin = feh.ge(lower) & (feh.le(upper) if i == len(axes) - 1 else feh.lt(upper))
    stars = df_master.loc[in_bin].dropna(subset=['Teff_kiel', 'L'])
    stars = stars.loc[stars['L'] > 0]

    ax.scatter(stars['Teff_kiel'], stars['L'], s=12, alpha=0.5)
    ax.set_title(f'[{lower:.2f}, {upper:.2f}] dex — N={len(stars)}')
    ax.set_xlim(5400, 4100)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(9*10**0, 5*10**2)

for ax in axes[3:]:
    ax.set_xlabel(r'$T_{\mathrm{eff}}$ [K]')
for ax in axes[::3]:
    ax.set_ylabel(r'Luminosity [$L_\odot$]')

fig.suptitle('HR diagrams by metallicity bin')
plt.tight_layout()
plt.show()

## Separating parts of the RGB and RC

### Willet separation

In [ ]:
### More masksss. This time to differentciate evolutionay states. We use Willet et al 2024 as reference

RGB_mask = (
    (df_master['L'] < 35)
    &(df_master['logg_willet_dnu'] > 2.6)
)

CHeB_mask = (
    (df_master['L'] >= 35)
    &(df_master['logg_willet_dnu'] < 2.5)
)

"""Ploting Willet separation --> realizing Willet separation does not include eAGB and therefore abandoning this way
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Kiel Diagram Colored by Metallicity [Fe/H] ---
ax1 = axes[0]
sc1 = ax1.scatter(
    kiel_sample_M_R.loc[RGB_mask ,'Teff_kiel'],
    kiel_sample_M_R.loc[RGB_mask, 'logg_willet_dnu'],
    c=kiel_sample_M_R.loc[RGB_mask,'feh'],
    cmap='viridis',
    s=15,
    alpha=0.7,
    edgecolors='none'
)

ax1.set_xlim(5400, 4100)  # Inverted Teff (hot on left, cool on right)
ax1.set_ylim(3.5, 1.0)    # Inverted logg (high logg at bottom, low at top)
ax1.set_xlabel(r'$T_{\mathrm{eff}}$ [K]', fontsize=12)
ax1.set_ylabel(r'$\log\,g$ [dex] (from M/R all dv constrained)', fontsize=12)
ax1.set_title(r'Kiel Diagram: $\log\,g$ vs. $T_{\mathrm{eff}}$ (colored by [Fe/H])', fontsize=13)
ax1.grid(True, alpha=0.3)
cb1 = plt.colorbar(sc1, ax=ax1)
cb1.set_label(r'$[\mathrm{Fe/H}]$ [dex]', fontsize=11)

# --- Panel 2: Kiel Diagram Colored by Density (Gaussian KDE) ---
ax2 = axes[1]
sc2 = ax2.scatter(
    kiel_sample_M_R.loc[CHeB_mask,'Teff_kiel'], 
    kiel_sample_M_R.loc[CHeB_mask,'logg_willet_dnu'],
    c=kiel_sample_M_R.loc[CHeB_mask,'feh'],
    cmap='magma',
    s=15,
    alpha=0.8,
    edgecolors='none'
)

ax2.set_xlim(5400, 4100)  # Inverted Teff
ax2.set_ylim(3.5, 1.0)    # Inverted logg
ax2.set_xlabel(r'$T_{\mathrm{eff}}$ [K]', fontsize=12)
ax2.set_ylabel(r'$\log\,g$ [dex] (from \nu$_max and Teff)', fontsize=12)
ax2.set_title(r'Kiel Diagram: $\log\,g$ vs. $T_{\mathrm{eff}}$ (colored by [Fe/H])', fontsize=13)
ax2.grid(True, alpha=0.3)
cb2 = plt.colorbar(sc2, ax=ax2)
cb2.set_label(r'$[\mathrm{Fe/H}]$ [dex]', fontsize=11)

plt.tight_layout()
plt.show()
"""
